Feature Step 1 — Load the clean data

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_parquet("electricity_clean.parquet")
print("loaded:", df.shape)
print(df.columns.tolist())

loaded: (11601895, 16)
['building_id', 'meter', 'timestamp', 'meter_reading', 'site_id', 'primary_use', 'square_feet', 'year_built', 'floor_count', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed']


Feature Step 2 — Calendar features

In [3]:
# Feature Step 2: calendar features from the timestamp
df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

df["hour"]       = df["timestamp"].dt.hour            # 0..23  (daily cycle)
df["weekday"]    = df["timestamp"].dt.dayofweek       # 0=Mon..6=Sun
df["month"]      = df["timestamp"].dt.month           # 1..12  (seasonality)
df["is_weekend"] = (df["weekday"] >= 5).astype("int8")  # 1 if Sat/Sun else 0

print(df[["timestamp", "hour", "weekday", "month", "is_weekend"]].head())
print("\nweekend share:", round(df["is_weekend"].mean(), 3))

            timestamp  hour  weekday  month  is_weekend
0 2016-01-30 08:00:00     8        5      1           1
1 2016-01-30 09:00:00     9        5      1           1
2 2016-01-30 10:00:00    10        5      1           1
3 2016-01-30 11:00:00    11        5      1           1
4 2016-01-30 12:00:00    12        5      1           1

weekend share: 0.286


Feature Step 3 — Lag features

In [4]:
# Feature Step 3: lag features (exact-timestamp)
base = df[["building_id", "timestamp", "meter_reading"]]

for hours, name in [(24, "lag_24h"), (168, "lag_168h")]:
    shifted = base.rename(columns={"meter_reading": name}).copy()
    shifted["timestamp"] = shifted["timestamp"] + pd.Timedelta(hours=hours)
    df = df.merge(shifted, on=["building_id", "timestamp"], how="left")

print(df[["timestamp", "meter_reading", "lag_24h", "lag_168h"]].head(3))
print("\nmissing lag_24h :", df["lag_24h"].isna().mean().round(3))
print("missing lag_168h:", df["lag_168h"].isna().mean().round(3))

            timestamp  meter_reading  lag_24h  lag_168h
0 2016-01-30 08:00:00      12.803751      NaN       NaN
1 2016-01-30 09:00:00       0.000000      NaN       NaN
2 2016-01-30 10:00:00       0.000000      NaN       NaN

missing lag_24h : 0.008
missing lag_168h: 0.032


Feature Step 4 — Rolling average 

In [5]:
# Feature Step 4: rolling mean of the PREVIOUS 24 hours (leakage-safe)
roll = (
    df.set_index("timestamp")
      .groupby("building_id")["meter_reading"]
      .rolling("24h", closed="left")     # window = prior 24h, EXCLUDING current row
      .mean()
      .reset_index(name="roll_mean_24h")
)
df = df.merge(roll, on=["building_id", "timestamp"], how="left")

print(df[["timestamp", "meter_reading", "lag_24h", "roll_mean_24h"]].head(3))
print("\nmissing roll_mean_24h:", df["roll_mean_24h"].isna().mean().round(3))

            timestamp  meter_reading  lag_24h  roll_mean_24h
0 2016-01-30 08:00:00      12.803751      NaN            NaN
1 2016-01-30 09:00:00       0.000000      NaN      12.803751
2 2016-01-30 10:00:00       0.000000      NaN       6.401875

missing roll_mean_24h: 0.0


Feature Step 5 — Building (metadata) features

In [6]:
# Feature Step 5: metadata features
# 1) size: log-transform square_feet (sizes are very skewed, like the target)
df["log_square_feet"] = np.log1p(df["square_feet"]).astype("float32")

# 2) primary_use is already a 'category' dtype -> LightGBM can use it directly
print("primary_use dtype:", df["primary_use"].dtype)

# 3) building_id: we keep it as a feature so the model can learn each
#    building's own baseline level (this is what makes it a 'global' model)
print("unique buildings:", df["building_id"].nunique())

df[["square_feet", "log_square_feet", "primary_use", "building_id"]].head(3)

primary_use dtype: str
unique buildings: 1413


,square_feet,log_square_feet,primary_use,building_id
0,7432,8.913685,Education,0
1,7432,8.913685,Education,0
2,7432,8.913685,Education,0


Feature Step 6: finalize features, drop unusable rows, save model-ready data

In [7]:
# make primary_use a real category so LightGBM can use it directly
df["primary_use"] = df["primary_use"].astype("category")

# columns we keep: keys + target + the features EDA justified
keep_cols = [
    "building_id", "timestamp", "meter_reading",         # keys + target
    "hour", "weekday", "month", "is_weekend",            # calendar
    "lag_24h", "lag_168h", "roll_mean_24h",              # history
    "air_temperature", "dew_temperature",                # weather
    "log_square_feet", "primary_use",                    # building
]
model_df = df[keep_cols].copy()

# drop rows missing any lag/rolling feature (the start of each building's series)
before = len(model_df)
model_df = model_df.dropna(subset=["lag_24h", "lag_168h", "roll_mean_24h"]).reset_index(drop=True)
print(f"rows: {before:,} -> {len(model_df):,}  (dropped {before-len(model_df):,} early-history rows)")

# save
model_df.to_parquet("model_ready.parquet", index=False)
print("\n✅ saved -> model_ready.parquet")
print("features:", [c for c in keep_cols if c not in ('building_id','timestamp','meter_reading')])
model_df.head(3)

rows: 11,601,895 -> 11,190,183  (dropped 411,712 early-history rows)

✅ saved -> model_ready.parquet
features: ['hour', 'weekday', 'month', 'is_weekend', 'lag_24h', 'lag_168h', 'roll_mean_24h', 'air_temperature', 'dew_temperature', 'log_square_feet', 'primary_use']


,building_id,timestamp,meter_reading,hour,weekday,month,is_weekend,lag_24h,lag_168h,roll_mean_24h,air_temperature,dew_temperature,log_square_feet,primary_use
0,0,2016-05-27 18:00:00,52.415363,18,4,5,0,53.415718,68.219910,53.098887,29.4,17.8,8.913685,Education
1,0,2016-05-27 19:00:00,51.615204,19,4,5,0,53.815796,81.423767,53.057206,30.6,17.8,8.913685,Education
2,0,2016-05-27 20:00:00,51.815098,20,4,5,0,51.214828,70.020416,52.965515,29.4,17.8,8.913685,Education
